In [1]:
# ==================================================
# RECUPERAR CONEXIÓN CON SQL SERVER
# ==================================================

import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from dotenv import load_dotenv

# Carpeta principal del proyecto
RUTA_PROYECTO = Path.cwd()

# Cargar configuración del proyecto
load_dotenv(RUTA_PROYECTO / ".env")

# Localizar el archivo database.py
RUTA_SRC = RUTA_PROYECTO / "src"

if str(RUTA_SRC) not in sys.path:
    sys.path.insert(0, str(RUTA_SRC))

# Importar conexión original
from database import engine_sqlserver

# Conectar con SQL Server
engine = engine_sqlserver()

with engine.connect() as conexion:
    nombre_bd = conexion.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

print("Conexión establecida correctamente.")
print("Base de datos:", nombre_bd)


Conexión establecida correctamente.
Base de datos: RedSaludCarazoDW_V2


In [3]:

# ==================================================
# PASO 13.1. CONTROL DE DISPONIBILIDAD DE DATOS
# ==================================================

print("=" * 65)
print("VALIDADOR DE CALIDAD - DISPONIBILIDAD TEMPORAL")
print("=" * 65)

# Verificar los años disponibles en los registros originales
consulta_raw = text("""
    SELECT
        YEAR(fecha_cita) AS anio,
        COUNT(*) AS registros_originales
    FROM dbo.raw_cita
    WHERE fecha_cita IS NOT NULL
    GROUP BY YEAR(fecha_cita)
    ORDER BY anio
""")

# Verificar los años disponibles en el dataset predictivo
consulta_predictiva = text("""
    SELECT
        YEAR(fecha_cita) AS anio,
        COUNT(*) AS registros_validados
    FROM dbo.dm_dataset_prediccion
    WHERE fecha_cita IS NOT NULL
    GROUP BY YEAR(fecha_cita)
    ORDER BY anio
""")

with engine.connect() as conexion:

    datos_raw = pd.read_sql_query(
        consulta_raw,
        conexion
    )

    datos_predictivos = pd.read_sql_query(
        consulta_predictiva,
        conexion
    )

print("\nREGISTROS ORIGINALES POR AÑO:")
display(datos_raw)

print("\nREGISTROS PREDICTIVOS POR AÑO:")
display(datos_predictivos)

print("\nVerificación de períodos completada.")


VALIDADOR DE CALIDAD - DISPONIBILIDAD TEMPORAL

REGISTROS ORIGINALES POR AÑO:


,anio,registros_originales
0,2026,10080



REGISTROS PREDICTIVOS POR AÑO:


,anio,registros_validados
0,2026,8996



Verificación de períodos completada.


In [7]:

# ============================================================
# PASO 24. EJECUCIÓN DE POLÍTICAS DE GOBIERNO
# ============================================================

import sys
import importlib
from pathlib import Path

from sqlalchemy import text
import pandas as pd

ruta_src = Path.cwd() / "src"

if str(ruta_src) not in sys.path:
    sys.path.insert(0, str(ruta_src))

import politicas_gobernanza

importlib.reload(politicas_gobernanza)

# Cargar registros originales desde SQL Server
with engine.connect() as conexion:

    datos_originales = pd.read_sql_query(
        text("SELECT * FROM dbo.raw_cita"),
        conexion
    )

# Evaluar políticas
tabla_politicas = (
    politicas_gobernanza.evaluar_politicas(
        datos_originales
    )
)

print("=" * 65)
print("EVALUACIÓN DE POLÍTICAS DE GOBIERNO DE DATOS")
print("=" * 65)

display(tabla_politicas)

print("\nRegistros revisados:", len(datos_originales))
print("Evaluación finalizada.")


EVALUACIÓN DE POLÍTICAS DE GOBIERNO DE DATOS


,Politica,Control,Incumplimientos,Estado
0,POL-02,Completitud,920.0,CON HALLAZGOS
1,POL-03,Estandarización,NaN,PENDIENTE DE CATÁLOGOS
2,POL-04,Duplicidad,80.0,REQUIERE REVISIÓN
3,POL-05,Valores anómalos,3.0,CON HALLAZGOS



Registros revisados: 10080
Evaluación finalizada.


In [8]:

# ============================================================
# PASO 25.1. DIAGNÓSTICO DE ESTANDARIZACIÓN
# POL-03 - CATÁLOGOS DE DATOS
# ============================================================

from sqlalchemy import text
import pandas as pd

print("=" * 65)
print("POL-03 - DIAGNÓSTICO DE CATEGORÍAS")
print("=" * 65)

columnas_catalogo = [
    "centro_salud",
    "especialidad",
    "canal_agendamiento"
]

with engine.connect() as conexion:

    for columna in columnas_catalogo:

        consulta = text(f"""
            SELECT
                {columna} AS valor,
                COUNT(*) AS cantidad
            FROM dbo.raw_cita
            GROUP BY {columna}
            ORDER BY cantidad DESC
        """)

        tabla = pd.read_sql_query(
            consulta,
            conexion
        )

        print("\n" + "-" * 65)
        print("CAMPO:", columna.upper())
        print("CATEGORÍAS ENCONTRADAS:", len(tabla))

        display(tabla)


POL-03 - DIAGNÓSTICO DE CATEGORÍAS

-----------------------------------------------------------------
CAMPO: CENTRO_SALUD
CATEGORÍAS ENCONTRADAS: 5


,valor,cantidad
0,JINOTEPE,3006
1,DIRIAMBA,2826
2,SAN MARCOS,2451
3,DOLORES,1677
4,NaN,120



-----------------------------------------------------------------
CAMPO: ESPECIALIDAD
CATEGORÍAS ENCONTRADAS: 6


,valor,cantidad
0,MEDICINA GENERAL,3332
1,LABORATORIO,2079
2,GINECOLOGÍA,1728
3,PEDIATRÍA,1520
4,CARDIOLOGÍA,1301
5,NaN,120



-----------------------------------------------------------------
CAMPO: CANAL_AGENDAMIENTO
CATEGORÍAS ENCONTRADAS: 5


,valor,cantidad
0,CALL CENTER,3463
1,WEB,2483
2,VENTANILLA,2478
3,REFERENCIA MÉDICA,1536
4,NaN,120


In [9]:

# ============================================================
# PASO 27. EVALUAR POLÍTICAS CON CATÁLOGOS DE REFERENCIA
# ============================================================

import json
import importlib
from pathlib import Path

import politicas_gobernanza

# Recargar el módulo
importlib.reload(politicas_gobernanza)

# Localizar archivo de categorías
ruta_catalogos = (
    Path.cwd() / "catalogos_propuestos_gobernanza.json"
)

# Leer categorías de referencia
with open(ruta_catalogos, "r", encoding="utf-8") as archivo:
    catalogos_referencia = json.load(archivo)

print("=" * 65)
print("POL-03 - VALIDACIÓN DE ESTANDARIZACIÓN")
print("=" * 65)

print("\nCatálogos de referencia cargados:")

for campo, valores in catalogos_referencia.items():
    print(f"- {campo}: {len(valores)} categorías")

# Ejecutar nuevamente las cuatro políticas
tabla_politicas_actualizada = (
    politicas_gobernanza.evaluar_politicas(
        datos_originales,
        catalogos=catalogos_referencia
    )
)

print("\nRESULTADO DE POLÍTICAS:")
display(tabla_politicas_actualizada)

print("\nNOTA:")
print("Catálogos provisionales basados en datos observados.")
print("Pendientes de aprobación institucional.")


POL-03 - VALIDACIÓN DE ESTANDARIZACIÓN

Catálogos de referencia cargados:
- centro_salud: 4 categorías
- especialidad: 5 categorías
- canal_agendamiento: 4 categorías

RESULTADO DE POLÍTICAS:


,Politica,Control,Incumplimientos,Estado
0,POL-02,Completitud,920,CON HALLAZGOS
1,POL-03,Estandarización,0,SIN HALLAZGOS
2,POL-04,Duplicidad,80,REQUIERE REVISIÓN
3,POL-05,Valores anómalos,3,CON HALLAZGOS



NOTA:
Catálogos provisionales basados en datos observados.
Pendientes de aprobación institucional.


In [10]:

# ============================================================
# EVALUAR POLÍTICAS DE GOBERNANZA ACTUALIZADAS
# ============================================================

import json
import importlib
from pathlib import Path

import politicas_gobernanza

# Cargar el código actualizado
importlib.reload(politicas_gobernanza)

# Leer catálogo provisional
ruta_catalogos = (
    Path.cwd() / "catalogos_propuestos_gobernanza.json"
)

with open(
    ruta_catalogos,
    "r",
    encoding="utf-8"
) as archivo:

    catalogos_referencia = json.load(
        archivo
    )

# Ejecutar las cuatro políticas
tabla_politicas_actualizada = (
    politicas_gobernanza.evaluar_politicas(
        datos_originales,
        catalogos=catalogos_referencia
    )
)

print("=" * 65)
print("RESULTADOS DE POLÍTICAS DE GOBIERNO DE DATOS")
print("=" * 65)

display(tabla_politicas_actualizada)

print("\nRegistros evaluados:", len(datos_originales))
print("Validación completada.")


RESULTADOS DE POLÍTICAS DE GOBIERNO DE DATOS


,Politica,Control,Incumplimientos,Estado
0,POL-02,Completitud,920,CON HALLAZGOS
1,POL-03,Estandarización,0,SIN HALLAZGOS - CATÁLOGO PROVISIONAL
2,POL-04,Duplicidad,80,REQUIERE REVISIÓN
3,POL-05,Valores anómalos,3,CON HALLAZGOS



Registros evaluados: 10080
Validación completada.


In [4]:

# ============================================================
# PASO 31. VISUALIZACIÓN DEL REPORTE DE AUDITORÍA
# RED SALUD INTEGRAL CARAZO
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

# Buscar el reporte de auditoría más reciente
carpeta = Path.cwd() / "resultados_jupyter"

archivos = sorted(
    carpeta.glob("auditoria_politicas_*.csv"),
    key=lambda archivo: archivo.stat().st_mtime
)

if not archivos:
    raise FileNotFoundError(
        "No se encontró el reporte de auditoría."
    )

# Cargar el archivo CSV
detalle_politicas = pd.read_csv(archivos[-1])

print("=" * 65)
print("RED SALUD INTEGRAL CARAZO")
print("REPORTE DE AUDITORÍA DE GOBERNANZA DE DATOS")
print("=" * 65)

print("\nRESUMEN DE INCUMPLIMIENTOS")

resumen = (
    detalle_politicas
    .groupby("politica")
    .agg(
        incidencias=("id_raw", "size"),
        filas_afectadas=("id_raw", "nunique")
    )
    .reset_index()
)

display(resumen)

# Mostrar únicamente los hallazgos de POL-02
print("\nPOL-02. REGISTROS INCOMPLETOS")

tabla_pol02 = detalle_politicas[
    detalle_politicas["politica"] == "POL-02"
].copy()

display(tabla_pol02.head(20))

print(
    "\nTotal de registros incompletos:",
    tabla_pol02["id_raw"].nunique()
)

print("\nArchivo consultado:", archivos[-1].name)


RED SALUD INTEGRAL CARAZO
REPORTE DE AUDITORÍA DE GOBERNANZA DE DATOS

RESUMEN DE INCUMPLIMIENTOS


,politica,incidencias,filas_afectadas
0,POL-02,920,920
1,POL-04,160,160
2,POL-05,3,3



POL-02. REGISTROS INCOMPLETOS


,id_raw,politica,campo,hallazgo
0,38,POL-02,centro_salud,Campo obligatorio faltante
1,330,POL-02,centro_salud,Campo obligatorio faltante
2,511,POL-02,centro_salud,Campo obligatorio faltante
3,556,POL-02,centro_salud,Campo obligatorio faltante
4,673,POL-02,centro_salud,Campo obligatorio faltante
5,675,POL-02,centro_salud,Campo obligatorio faltante
6,680,POL-02,centro_salud,Campo obligatorio faltante
7,683,POL-02,centro_salud,Campo obligatorio faltante
8,766,POL-02,centro_salud,Campo obligatorio faltante
9,795,POL-02,centro_salud,Campo obligatorio faltante



Total de registros incompletos: 920

Archivo consultado: auditoria_politicas_20261019T205327Z.csv


In [16]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# PASO 30. CONTROL DE CALIDAD PREVIO AL ENTRENAMIENTO
# Integración de Gobernanza con Minería de Datos
# ============================================================

import pandas as pd
from sqlalchemy import text


# ============================================================
# 1. CONFIGURACIÓN GENERAL
# ============================================================

ANIO_MODELO = 2024

COLUMNAS_REQUERIDAS = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados",
    "no_asistio"
]

COLUMNAS_NUMERICAS = [
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados"
]

COLUMNAS_TEXTO = [
    "id_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad"
]


print("=" * 65)
print("RED SALUD INTEGRAL CARAZO")
print("CONTROL DE CALIDAD PREVIO A MINERÍA DE DATOS")
print("=" * 65)


# ============================================================
# 2. CONSULTAR EL DATA WAREHOUSE
# ============================================================

consulta = text("""
    SELECT *
    FROM dbo.dm_dataset_prediccion
    WHERE fecha_cita >= :fecha_inicio
      AND fecha_cita < :fecha_fin
""")

with engine.connect() as conexion:

    datos_control = pd.read_sql_query(
        consulta,
        conexion,
        params={
            "fecha_inicio": f"{ANIO_MODELO}-01-01",
            "fecha_fin": f"{ANIO_MODELO + 1}-01-01"
        }
    )

print("\nCONEXIÓN CON SQL SERVER")
print("Tabla: dbo.dm_dataset_prediccion")
print("Año solicitado:", ANIO_MODELO)
print("Registros encontrados:", len(datos_control))


# ============================================================
# 3. INICIALIZAR HALLAZGOS
# ============================================================

hallazgos_control = []


# ============================================================
# 4. VERIFICAR DISPONIBILIDAD
# ============================================================

if datos_control.empty:

    hallazgos_control.append(
        "No existen registros predictivos "
        f"para el año {ANIO_MODELO}."
    )


# ============================================================
# 5. VERIFICAR COLUMNAS REQUERIDAS
# ============================================================

faltantes = (
    set(COLUMNAS_REQUERIDAS)
    - set(datos_control.columns)
)

if faltantes:

    hallazgos_control.append(
        f"Faltan columnas obligatorias: "
        f"{sorted(faltantes)}"
    )


# ============================================================
# 6. EJECUTAR VALIDACIONES DE CALIDAD
# ============================================================

if not datos_control.empty and not faltantes:

    # --------------------------------------------------------
    # POL-02. COMPLETITUD
    # --------------------------------------------------------

    print("\nPOL-02. VALIDACIÓN DE COMPLETITUD")

    for campo in COLUMNAS_REQUERIDAS:

        cantidad_nulos = int(
            datos_control[campo].isna().sum()
        )

        if cantidad_nulos > 0:

            hallazgos_control.append(
                f"POL-02: {campo} contiene "
                f"{cantidad_nulos} valores nulos."
            )

    # Verificar cadenas vacías
    for campo in COLUMNAS_TEXTO:

        vacios = (
            datos_control[campo]
            .astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        )

        cantidad_vacios = int(vacios.sum())

        if cantidad_vacios > 0:

            hallazgos_control.append(
                f"POL-02: {campo} contiene "
                f"{cantidad_vacios} valores vacíos."
            )

    print("Comprobación de campos finalizada.")

    # --------------------------------------------------------
    # POL-04. CONTROL DE DUPLICIDAD
    # --------------------------------------------------------

    print("\nPOL-04. CONTROL DE DUPLICIDAD")

    ids = (
        datos_control["id_cita"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

    duplicados = (
        ids.notna()
        & ids.duplicated(keep=False)
    )

    grupos_duplicados = int(
        ids[duplicados].nunique()
    )

    if grupos_duplicados > 0:

        hallazgos_control.append(
            "POL-04: Se encontraron "
            f"{grupos_duplicados} identificadores "
            "repetidos."
        )

    print("Grupos repetidos:", grupos_duplicados)

    # --------------------------------------------------------
    # POL-05. CONTROL DE VALORES NUMÉRICOS
    # --------------------------------------------------------

    print("\nPOL-05. VALIDACIÓN DE VALORES")

    for campo in COLUMNAS_NUMERICAS:

        valores = pd.to_numeric(
            datos_control[campo],
            errors="coerce"
        )

        invalidos = (
            valores.isna()
            | valores.lt(0)
            | valores.ne(valores.round())
        )

        cantidad_invalidos = int(
            invalidos.fillna(True).sum()
        )

        if cantidad_invalidos > 0:

            hallazgos_control.append(
                f"POL-05: {campo} contiene "
                f"{cantidad_invalidos} valores inválidos."
            )

    # --------------------------------------------------------
    # 7. VALIDAR VARIABLE OBJETIVO
    # --------------------------------------------------------

    objetivo_control = pd.to_numeric(
        datos_control["no_asistio"],
        errors="coerce"
    )

    objetivo_valido = (
        objetivo_control.isin([0, 1])
    )

    cantidad_objetivos_invalidos = int(
        (~objetivo_valido).sum()
    )

    if cantidad_objetivos_invalidos > 0:

        hallazgos_control.append(
            "POL-05: La variable no_asistio "
            f"contiene {cantidad_objetivos_invalidos} "
            "valores inválidos."
        )

    # --------------------------------------------------------
    # 8. VALIDAR FECHAS
    # --------------------------------------------------------

    fechas_control = pd.to_datetime(
        datos_control["fecha_cita"],
        errors="coerce"
    )

    fechas_invalidas = int(
        fechas_control.isna().sum()
    )

    if fechas_invalidas > 0:

        hallazgos_control.append(
            f"Existen {fechas_invalidas} "
            "fechas de cita inválidas."
        )

    # --------------------------------------------------------
    # 9. COMPROBAR PARTICIÓN TEMPORAL
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("VERIFICACIÓN DE PARTICIÓN TEMPORAL")
    print("=" * 65)

    mascara_train = (
        (fechas_control >= f"{ANIO_MODELO}-01-01")
        &
        (fechas_control < f"{ANIO_MODELO}-05-01")
    )

    mascara_val = (
        (fechas_control >= f"{ANIO_MODELO}-05-01")
        &
        (fechas_control < f"{ANIO_MODELO}-06-01")
    )

    mascara_test = (
        (fechas_control >= f"{ANIO_MODELO}-06-01")
        &
        (fechas_control < f"{ANIO_MODELO}-07-01")
    )

    particiones = [
        ("Entrenamiento", mascara_train),
        ("Validación", mascara_val),
        ("Prueba final", mascara_test)
    ]

    for nombre, mascara in particiones:

        cantidad = int(mascara.sum())

        print(f"{nombre}: {cantidad} registros")

        if cantidad == 0:

            hallazgos_control.append(
                f"El conjunto {nombre} está vacío."
            )

        elif objetivo_control[mascara].nunique() < 2:

            hallazgos_control.append(
                f"El conjunto {nombre} "
                "no contiene ambas clases."
            )

    # Verificar registros fuera de enero-junio
    fuera_periodo = ~(
        mascara_train
        | mascara_val
        | mascara_test
    )

    cantidad_fuera = int(
        fuera_periodo.sum()
    )

    if cantidad_fuera > 0:

        hallazgos_control.append(
            f"Existen {cantidad_fuera} registros "
            "fuera del período enero-junio."
        )


# ============================================================
# 10. DETERMINAR ESTADO DEL CONTROL
# ============================================================

control_apto = len(hallazgos_control) == 0

print("\n" + "=" * 65)
print("RESULTADO DEL CONTROL DE CALIDAD")
print("=" * 65)

print("Registros evaluados:", len(datos_control))
print("Hallazgos detectados:", len(hallazgos_control))


if control_apto:

    print("\nESTADO: APTO")

    print(
        "El dataset superó las comprobaciones "
        "esenciales de calidad."
    )

    print(
        "\nAUTORIZADO: continuar con "
        "Minería de Datos."
    )

else:

    print("\nESTADO: NO APTO")

    print("\nHALLAZGOS IDENTIFICADOS:")

    for numero, hallazgo in enumerate(
        hallazgos_control,
        start=1
    ):

        print(f"{numero}. {hallazgo}")

    print(
        "\nNo se autoriza el entrenamiento "
        "con estos datos."
    )



# ============================================================
# 11. CONTROLAR LA CONTINUIDAD DEL PIPELINE
# ============================================================

if control_apto:

    print("\n" + "=" * 65)
    print("VALIDACIÓN FINALIZADA CORRECTAMENTE")
    print("=" * 65)

    print("ESTADO: APTO")
    print("El período solicitado cumple las")
    print("condiciones esenciales de calidad.")

    print("\nDECISIÓN DEL PIPELINE:")
    print("Se permite continuar con minería de datos.")

else:

    print("\n" + "=" * 65)
    print("VALIDACIÓN FINALIZADA CON HALLAZGOS")
    print("=" * 65)

    print("ESTADO: NO APTO")

    print("\nDECISIÓN DEL PIPELINE:")
    print("No se autoriza el entrenamiento")
    print("para el período solicitado.")

    print("\nACCIÓN RECOMENDADA:")
    print("Revisar los hallazgos y comprobar")
    print("la disponibilidad y calidad de los datos.")

print("\nLa validación terminó sin errores de ejecución.")



RED SALUD INTEGRAL CARAZO
CONTROL DE CALIDAD PREVIO A MINERÍA DE DATOS

CONEXIÓN CON SQL SERVER
Tabla: dbo.dm_dataset_prediccion
Año solicitado: 2024
Registros encontrados: 0

RESULTADO DEL CONTROL DE CALIDAD
Registros evaluados: 0
Hallazgos detectados: 1

ESTADO: NO APTO

HALLAZGOS IDENTIFICADOS:
1. No existen registros predictivos para el año 2024.

No se autoriza el entrenamiento con estos datos.

VALIDACIÓN FINALIZADA CON HALLAZGOS
ESTADO: NO APTO

DECISIÓN DEL PIPELINE:
No se autoriza el entrenamiento
para el período solicitado.

ACCIÓN RECOMENDADA:
Revisar los hallazgos y comprobar
la disponibilidad y calidad de los datos.

La validación terminó sin errores de ejecución.


In [5]:

# ============================================================
# PASO 32. DEMOSTRACIÓN DE TRAZABILIDAD
# RED SALUD INTEGRAL CARAZO - POL-02
# ============================================================

import pandas as pd
from pathlib import Path
from sqlalchemy import text
from IPython.display import display

# Identificador del registro que vamos a investigar
ID_EJEMPLO = 38

print("=" * 65)
print("TRAZABILIDAD DE DATOS - POL-02")
print("=" * 65)

# ------------------------------------------------------------
# 1. CONSULTAR EL REGISTRO ORIGINAL EN SQL SERVER
# ------------------------------------------------------------

consulta = text("""
    SELECT
        id_raw,
        id_cita,
        centro_salud,
        origen_dataset,
        sistema_origen,
        lote_carga,
        fecha_ingesta
    FROM dbo.raw_cita
    WHERE id_raw = :id_raw
""")

with engine.connect() as conexion:
    registro_origen = pd.read_sql_query(
        consulta,
        conexion,
        params={"id_raw": ID_EJEMPLO}
    )

print("\n1. REGISTRO EN SQL SERVER")
display(registro_origen)

# ------------------------------------------------------------
# 2. LOCALIZAR SU HALLAZGO EN EL REPORTE DE AUDITORÍA
# ------------------------------------------------------------

carpeta = Path.cwd() / "resultados_jupyter"

archivos = sorted(
    carpeta.glob("auditoria_politicas_*.csv"),
    key=lambda archivo: archivo.stat().st_mtime
)

if not archivos:
    raise FileNotFoundError(
        "No se encontró el reporte de auditoría."
    )

auditoria = pd.read_csv(archivos[-1])

hallazgos_registro = auditoria[
    (auditoria["id_raw"] == ID_EJEMPLO)
    & (auditoria["politica"] == "POL-02")
].copy()

print("\n2. HALLAZGO IDENTIFICADO POR EL VALIDADOR")
display(hallazgos_registro)

# ------------------------------------------------------------
# 3. COMPROBAR EL HALLAZGO EN EL REGISTRO ORIGINAL
# ------------------------------------------------------------

print("\n3. COMPROBACIÓN DE TRAZABILIDAD")

if registro_origen.empty:
    print("No se encontró el registro original.")

elif hallazgos_registro.empty:
    print("No existe un hallazgo POL-02 para este registro.")

else:
    for campo in hallazgos_registro["campo"].unique():

        valor = registro_origen.iloc[0][campo]

        esta_incompleto = (
            pd.isna(valor)
            or (
                isinstance(valor, str)
                and valor.strip() == ""
            )
        )

        print("Identificador interno:", ID_EJEMPLO)
        print("Política evaluada: POL-02")
        print("Campo evaluado:", campo)
        print("Valor en SQL Server:", repr(valor))

        if esta_incompleto:
            print("COMPROBACIÓN: HALLAZGO CONFIRMADO")
        else:
            print("COMPROBACIÓN: REQUIERE REVISIÓN")

print("\nArchivo de auditoría utilizado:")
print(archivos[-1].name)


TRAZABILIDAD DE DATOS - POL-02

1. REGISTRO EN SQL SERVER


,id_raw,id_cita,centro_salud,origen_dataset,sistema_origen,lote_carga,fecha_ingesta
0,38,RSV2-000038,None,ampliado,HIS,20260928T002744032220Z,2026-09-28 00:27:45.112971



2. HALLAZGO IDENTIFICADO POR EL VALIDADOR


,id_raw,politica,campo,hallazgo
0,38,POL-02,centro_salud,Campo obligatorio faltante



3. COMPROBACIÓN DE TRAZABILIDAD
Identificador interno: 38
Política evaluada: POL-02
Campo evaluado: centro_salud
Valor en SQL Server: None
COMPROBACIÓN: HALLAZGO CONFIRMADO

Archivo de auditoría utilizado:
auditoria_politicas_20261019T205327Z.csv


In [8]:

# ============================================================
# PASO 33. TRAZABILIDAD DEL ORIGEN
# EXCEL -> SQL SERVER
# RED SALUD INTEGRAL CARAZO
# ============================================================

import pandas as pd
from pathlib import Path
from sqlalchemy import text
from IPython.display import display

print("=" * 65)
print("TRAZABILIDAD: EXCEL -> SQL SERVER")
print("=" * 65)

# ============================================================
# 1. LOCALIZAR EL ARCHIVO EXCEL
# ============================================================

nombre_excel = "C02_Citas_v3.xlsx"

rutas_posibles = [
    Path.cwd() / nombre_excel,
    Path.cwd() / "RedSaludCarazo_proyecto_base" / nombre_excel,
    Path.cwd().parent / nombre_excel
]

ruta_excel = next(
    (ruta for ruta in rutas_posibles if ruta.is_file()),
    None
)

if ruta_excel is None:
    raise FileNotFoundError(
        "No se encontró C02_Citas_v3.xlsx. "
        "Comprueba que esté en la carpeta del proyecto."
    )

# ============================================================
# 2. LEER LOS REGISTROS DEL EXCEL
# ============================================================

datos_excel = pd.read_excel(ruta_excel)

print("\n1. ARCHIVO ORIGINAL")
print("Archivo:", ruta_excel.name)
print("Registros:", len(datos_excel))
print("Columnas:", len(datos_excel.columns))

# ============================================================
# 3. CONSULTAR LOS REGISTROS DE SQL SERVER
# ============================================================

with engine.connect() as conexion:

    registros_sql = pd.read_sql_query(
        text("""
            SELECT
                id_raw,
                id_cita,
                origen_dataset,
                lote_carga,
                fecha_ingesta
            FROM dbo.raw_cita
        """),
        conexion
    )

print("\n2. TABLA ORIGINAL EN SQL SERVER")
print("Tabla: dbo.raw_cita")
print("Registros:", len(registros_sql))

# ============================================================
# 4. COMPARAR IDENTIFICADORES Y REPETICIONES
# ============================================================

if "id_cita" not in datos_excel.columns:
    raise ValueError(
        "El archivo Excel no contiene id_cita."
    )

def normalizar_ids(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.upper()
        .fillna("<SIN_ID>")
        .replace("", "<SIN_ID>")
    )

ids_excel = normalizar_ids(datos_excel["id_cita"])
ids_sql = normalizar_ids(registros_sql["id_cita"])

# Se comparan las frecuencias de los identificadores,
# para considerar también posibles duplicados.

conteo_excel = ids_excel.value_counts(dropna=False)
conteo_sql = ids_sql.value_counts(dropna=False)

comparacion_ids = pd.concat(
    [
        conteo_excel.rename("excel"),
        conteo_sql.rename("sql_server")
    ],
    axis=1
).fillna(0).astype(int)

diferencias = comparacion_ids[
    comparacion_ids["excel"] != comparacion_ids["sql_server"]
]

# ============================================================
# 5. PRESENTAR EVIDENCIA DE TRAZABILIDAD
# ============================================================

resumen_trazabilidad = pd.DataFrame([
    {
        "Comprobación": "Cantidad de registros",
        "Excel": len(datos_excel),
        "SQL Server": len(registros_sql)
    },
    {
        "Comprobación": "Identificadores distintos",
        "Excel": int(ids_excel.nunique()),
        "SQL Server": int(ids_sql.nunique())
    },
    {
        "Comprobación": "Identificadores repetidos (grupos)",
        "Excel": int((conteo_excel > 1).sum()),
        "SQL Server": int((conteo_sql > 1).sum())
    }
])

print("\n3. COMPARACIÓN DE LAS FUENTES")
display(resumen_trazabilidad)

print("\nIdentificadores con diferencias de frecuencia:")
print(len(diferencias))

if diferencias.empty:
    print(
        "Los identificadores y sus frecuencias "
        "coinciden entre el Excel y SQL Server."
    )
else:
    print("Se encontraron diferencias entre las fuentes.")
    display(diferencias.head(10))

# ============================================================
# 6. COMPROBAR INFORMACIÓN DE LA CARGA
# ============================================================

print("\n4. PROCEDENCIA Y LOTES DE CARGA")

lotes = (
    registros_sql
    .groupby(
        ["origen_dataset", "lote_carga"],
        dropna=False
    )
    .size()
    .reset_index(name="registros")
)

display(lotes)

# ============================================================
# 7. RESULTADO
# ============================================================

coinciden_identificadores = diferencias.empty

if coinciden_identificadores:
    print("\nRESULTADO:")
    print("Coincidencia de identificadores: CONFIRMADA")
else:
    print("\nRESULTADO:")
    print("Coincidencia de identificadores: REQUIERE REVISIÓN")

print("\nIMPORTANTE:")
print(
    "La coincidencia de identificadores no demuestra "
    "por sí sola que todos los campos sean idénticos."
)
print(
    "Las transformaciones y los valores de las columnas "
    "se comprobarán en las siguientes etapas."
)


TRAZABILIDAD: EXCEL -> SQL SERVER

1. ARCHIVO ORIGINAL
Archivo: C02_Citas_v3.xlsx
Registros: 10080
Columnas: 15

2. TABLA ORIGINAL EN SQL SERVER
Tabla: dbo.raw_cita
Registros: 10080

3. COMPARACIÓN DE LAS FUENTES


,Comprobación,Excel,SQL Server
0,Cantidad de registros,10080,10080
1,Identificadores distintos,10000,10000
2,Identificadores repetidos (grupos),80,80



Identificadores con diferencias de frecuencia:
0
Los identificadores y sus frecuencias coinciden entre el Excel y SQL Server.

4. PROCEDENCIA Y LOTES DE CARGA


,origen_dataset,lote_carga,registros
0,ampliado,20260928T002744032220Z,10080



RESULTADO:
Coincidencia de identificadores: CONFIRMADA

IMPORTANTE:
La coincidencia de identificadores no demuestra por sí sola que todos los campos sean idénticos.
Las transformaciones y los valores de las columnas se comprobarán en las siguientes etapas.


In [9]:

# ============================================================
# PASO 34. TRAZABILIDAD DEL DATA WAREHOUSE
# dbo.raw_cita -> dbo.dm_dataset_prediccion
# RED SALUD INTEGRAL CARAZO
# ============================================================

import pandas as pd
from sqlalchemy import text
from IPython.display import display

print("=" * 65)
print("TRAZABILIDAD: DATOS ORIGINALES -> DATASET PREDICTIVO")
print("=" * 65)

# ============================================================
# 1. CONSULTAR AMBAS TABLAS EN SQL SERVER
# ============================================================

with engine.connect() as conexion:

    raw = pd.read_sql_query(
        text("""
            SELECT id_raw, id_cita
            FROM dbo.raw_cita
        """),
        conexion
    )

    predictivo = pd.read_sql_query(
        text("""
            SELECT id_cita, fecha_cita
            FROM dbo.dm_dataset_prediccion
        """),
        conexion
    )

# ============================================================
# 2. NORMALIZAR LOS IDENTIFICADORES
# ============================================================

def normalizar_identificador(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

raw["id_normalizado"] = normalizar_identificador(
    raw["id_cita"]
)

predictivo["id_normalizado"] = normalizar_identificador(
    predictivo["id_cita"]
)

# ============================================================
# 3. IDENTIFICAR CORRESPONDENCIAS
# ============================================================

ids_raw = set(
    raw["id_normalizado"].dropna()
)

ids_predictivo = set(
    predictivo["id_normalizado"].dropna()
)

ids_compartidos = ids_raw & ids_predictivo

ids_solo_raw = ids_raw - ids_predictivo

ids_solo_predictivo = ids_predictivo - ids_raw

# Filas originales cuyo identificador no está
# presente en el dataset predictivo
filas_sin_correspondencia = raw[
    raw["id_normalizado"].isin(ids_solo_raw)
].copy()

# Identificar duplicados en cada tabla
grupos_duplicados_raw = int(
    (
        raw["id_normalizado"]
        .dropna()
        .value_counts() > 1
    ).sum()
)

grupos_duplicados_predictivo = int(
    (
        predictivo["id_normalizado"]
        .dropna()
        .value_counts() > 1
    ).sum()
)

# ============================================================
# 4. PRESENTAR EL RESUMEN
# ============================================================

resumen = pd.DataFrame([
    {
        "Comprobación": "Total de registros",
        "Datos originales": len(raw),
        "Dataset predictivo": len(predictivo)
    },
    {
        "Comprobación": "Identificadores distintos",
        "Datos originales": len(ids_raw),
        "Dataset predictivo": len(ids_predictivo)
    },
    {
        "Comprobación": "Grupos de IDs repetidos",
        "Datos originales": grupos_duplicados_raw,
        "Dataset predictivo": grupos_duplicados_predictivo
    }
])

print("\n1. COMPARACIÓN DE LAS TABLAS")
display(resumen)

print("\n2. CORRESPONDENCIA DE IDENTIFICADORES")

print(
    "Identificadores presentes en ambas tablas:",
    len(ids_compartidos)
)

print(
    "Identificadores únicamente en raw_cita:",
    len(ids_solo_raw)
)

print(
    "Identificadores únicamente en el predictivo:",
    len(ids_solo_predictivo)
)

print(
    "Filas originales cuyos IDs no aparecen "
    "en el predictivo:",
    len(filas_sin_correspondencia)
)

print(
    "Diferencia total entre cantidades de filas:",
    len(raw) - len(predictivo)
)

# ============================================================
# 5. MOSTRAR EJEMPLOS PARA LA AUDITORÍA
# ============================================================

print("\n3. EJEMPLOS DE REGISTROS SIN CORRESPONDENCIA")

display(
    filas_sin_correspondencia[
        ["id_raw", "id_cita"]
    ].head(10)
)

# ============================================================
# 6. COMPROBAR PROCEDENCIA DEL DATASET PREDICTIVO
# ============================================================

print("\n4. RESULTADO DE TRAZABILIDAD")

if len(ids_solo_predictivo) == 0:
    print(
        "CONFIRMADO: Todos los identificadores "
        "del dataset predictivo aparecen "
        "en los datos originales."
    )
else:
    print(
        "REQUIERE REVISIÓN: Existen identificadores "
        "predictivos que no aparecen en raw_cita."
    )

print("\nIMPORTANTE:")
print(
    "Esta comprobación establece correspondencia "
    "por identificador, no las causas de exclusión "
    "ni la igualdad de todos los valores."
)

print("\nConsulta realizada sin modificar SQL Server.")


TRAZABILIDAD: DATOS ORIGINALES -> DATASET PREDICTIVO

1. COMPARACIÓN DE LAS TABLAS


,Comprobación,Datos originales,Dataset predictivo
0,Total de registros,10080,8996
1,Identificadores distintos,10000,8996
2,Grupos de IDs repetidos,80,0



2. CORRESPONDENCIA DE IDENTIFICADORES
Identificadores presentes en ambas tablas: 8996
Identificadores únicamente en raw_cita: 1004
Identificadores únicamente en el predictivo: 0
Filas originales cuyos IDs no aparecen en el predictivo: 1084
Diferencia total entre cantidades de filas: 1084

3. EJEMPLOS DE REGISTROS SIN CORRESPONDENCIA


,id_raw,id_cita
2,3,RSV2-000003
4,5,RSV2-000005
15,16,RSV2-000016
17,18,RSV2-000018
19,20,RSV2-000020
37,38,RSV2-000038
41,42,RSV2-000042
67,68,RSV2-000068
76,77,RSV2-000077
82,83,RSV2-000083



4. RESULTADO DE TRAZABILIDAD
CONFIRMADO: Todos los identificadores del dataset predictivo aparecen en los datos originales.

IMPORTANTE:
Esta comprobación establece correspondencia por identificador, no las causas de exclusión ni la igualdad de todos los valores.

Consulta realizada sin modificar SQL Server.


In [3]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# EVALUACIÓN DE LOS 3 KPI DE CALIDAD DE DATOS
# ============================================================

import pandas as pd
from sqlalchemy import text
from IPython.display import display

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("EVALUACIÓN DE LOS TRES KPI DE CALIDAD")
print("=" * 70)


# ============================================================
# 1. CONSULTAR LOS DATOS DESDE SQL SERVER
# ============================================================

with engine.connect() as conexion:

    # Registros originales
    datos = pd.read_sql_query(
        text("SELECT * FROM dbo.raw_cita"),
        conexion
    )

    # Registros preparados para minería de datos
    datos_predictivos = pd.read_sql_query(
        text("SELECT * FROM dbo.dm_dataset_prediccion"),
        conexion
    )

total_registros = len(datos)
total_predictivos = len(datos_predictivos)

if total_registros == 0:
    raise ValueError(
        "No existen registros originales para evaluar."
    )

print("\nDATOS CONSULTADOS")
print("Registros originales:", total_registros)
print("Registros predictivos:", total_predictivos)


# ============================================================
# KPI 1. PORCENTAJE DE REGISTROS VALIDADOS (PRV)
# ============================================================
#
# Fórmula:
# PRV = (Registros en dataset predictivo /
#        Total de registros originales) * 100
#
# Meta: >= 90%
#
# Este KPI mide la proporción incorporada al dataset
# predictivo. No demuestra por sí solo que todas
# las exclusiones sean errores de calidad.
# ============================================================

registros_validados = total_predictivos

prv = (
    registros_validados / total_registros
) * 100

meta_prv = 90

estado_prv = (
    "CUMPLE"
    if prv >= meta_prv
    else "NO CUMPLE"
)


# ============================================================
# KPI 2. TASA DE INTEGRIDAD DE REGISTROS DE CITAS
# ============================================================
#
# Fórmula:
# TID = (Registros completos /
#        Total de registros procesados) * 100
#
# Meta del II corte: > 98%
#
# ============================================================

campos_obligatorios = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados",
    "no_asistio"
]

# Verificar existencia de columnas
faltan_columnas = (
    set(campos_obligatorios) - set(datos.columns)
)

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas: {sorted(faltan_columnas)}"
    )

# Detectar valores nulos
faltantes = datos[campos_obligatorios].isna().copy()

# Detectar también textos vacíos
for campo in campos_obligatorios:

    serie = datos[campo]

    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):

        vacios = (
            serie.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        )

        faltantes[campo] = (
            faltantes[campo] | vacios
        )

# Un registro es incompleto si falta
# al menos uno de los campos obligatorios
registros_incompletos = int(
    faltantes.any(axis=1).sum()
)

registros_completos = (
    total_registros - registros_incompletos
)

tasa_integridad = (
    registros_completos / total_registros
) * 100

meta_integridad = 98

estado_integridad = (
    "CUMPLE"
    if tasa_integridad > meta_integridad
    else "NO CUMPLE"
)


# ============================================================
# KPI 3. TASA DE DUPLICIDAD DE IDENTIFICADORES DE CITA
# ============================================================
#
# Fórmula del II corte:
# (IDs de citas repetidos / Total de registros) * 100
#
# Se cuentan los identificadores distintos
# que aparecen más de una vez.
#
# Meta: 0%
#
# ============================================================

ids = (
    datos["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

# Contar frecuencia de cada identificador
frecuencias = ids.dropna().value_counts()

# Identificadores que aparecen más de una vez
ids_repetidos = int(
    (frecuencias > 1).sum()
)

# Filas involucradas en los grupos duplicados
filas_duplicadas = int(
    frecuencias[frecuencias > 1].sum()
)

tasa_duplicidad = (
    ids_repetidos / total_registros
) * 100

meta_duplicidad = 0

estado_duplicidad = (
    "CUMPLE"
    if ids_repetidos == 0
    else "NO CUMPLE"
)


# ============================================================
# 4. CONSTRUIR TABLA GENERAL DE KPI
# ============================================================

tabla_3_kpis = pd.DataFrame([
    {
        "Código": "PRV",
        "KPI": "Porcentaje de Registros Validados",
        "Resultado (%)": round(prv, 2),
        "Meta": ">=90%",
        "Estado": estado_prv
    },
    {
        "Código": "IND-01",
        "KPI": "Tasa de integridad de registros de citas",
        "Resultado (%)": round(tasa_integridad, 2),
        "Meta": ">98%",
        "Estado": estado_integridad
    },
    {
        "Código": "IND-05",
        "KPI": "Tasa de duplicidad de identificadores de cita",
        "Resultado (%)": round(tasa_duplicidad, 2),
        "Meta": "0%",
        "Estado": estado_duplicidad
    }
])


# ============================================================
# 5. MOSTRAR RESULTADOS
# ============================================================

print("\n" + "=" * 70)
print("DETALLE DE LOS INDICADORES")
print("=" * 70)

print("\nKPI 1. PORCENTAJE DE REGISTROS VALIDADOS")
print("Registros originales:", total_registros)
print("Registros predictivos:", registros_validados)
print(f"Resultado: {prv:.2f}%")
print("Estado:", estado_prv)

print("\nKPI 2. TASA DE INTEGRIDAD")
print("Registros completos:", registros_completos)
print("Registros incompletos:", registros_incompletos)
print(f"Resultado: {tasa_integridad:.2f}%")
print("Estado:", estado_integridad)

print("\nKPI 3. TASA DE DUPLICIDAD")
print("Identificadores repetidos:", ids_repetidos)
print("Filas involucradas:", filas_duplicadas)
print(f"Resultado: {tasa_duplicidad:.2f}%")
print("Estado:", estado_duplicidad)

print("\n" + "=" * 70)
print("TABLA GENERAL DE LOS 3 KPI DE CALIDAD")
print("=" * 70)

display(tabla_3_kpis)

print("\nEvaluación finalizada.")
print("No se modificaron los datos de SQL Server.")


RED SALUD INTEGRAL CARAZO
EVALUACIÓN DE LOS TRES KPI DE CALIDAD

DATOS CONSULTADOS
Registros originales: 10080
Registros predictivos: 8996

DETALLE DE LOS INDICADORES

KPI 1. PORCENTAJE DE REGISTROS VALIDADOS
Registros originales: 10080
Registros predictivos: 8996
Resultado: 89.25%
Estado: NO CUMPLE

KPI 2. TASA DE INTEGRIDAD
Registros completos: 9160
Registros incompletos: 920
Resultado: 90.87%
Estado: NO CUMPLE

KPI 3. TASA DE DUPLICIDAD
Identificadores repetidos: 80
Filas involucradas: 160
Resultado: 0.79%
Estado: NO CUMPLE

TABLA GENERAL DE LOS 3 KPI DE CALIDAD


,Código,KPI,Resultado (%),Meta,Estado
0,PRV,Porcentaje de Registros Validados,89.25,>=90%,NO CUMPLE
1,IND-01,Tasa de integridad de registros de citas,90.87,>98%,NO CUMPLE
2,IND-05,Tasa de duplicidad de identificadores de cita,0.79,0%,NO CUMPLE



Evaluación finalizada.
No se modificaron los datos de SQL Server.
